In [1]:
ROOT_PATH = 'C:/Users/khoan/OneDrive/Documents/stock_data_scraper'
import os
os.chdir(ROOT_PATH)

In [2]:
from utils.utilities import get_engine, correct_nan_val
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import datetime, timedelta
from sqlalchemy import text
import collections

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, LSTM, Dropout, Attention, Input, Conv1D
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import regularizers

In [3]:
import warnings
# Ignore all warnings
warnings.filterwarnings("ignore")

In [4]:
import pickle
with open("model/base_stocks.pkl", "rb") as f:
    # Load the variable from the file
    base_stocks = pickle.load(f)

In [5]:
STOCK_CODES = ['PPT','TPG']
FEATURES = ['open', 'high', 'low', 'volume', 'close']

In [6]:
# get engine
engine = get_engine(country = 'AU')
days = 365 * 5
time_span = 60

In [ ]:
stock_code = STOCK_CODES['PPT']

In [13]:
# Collect data for each stock
def collect_data(stock_codes, days = 3650, time_span = 60):
    data_dict = {}
    standard_dict = collections.defaultdict(dict)
    for stock_code in stock_codes:
        if days != 'max':
            query = f"""
                SELECT
                    date,
                    open,
                    high,
                    low,
                    close,
                    volume
                FROM transaction
                WHERE 
                    stock_code = '{stock_code}'
                ORDER BY date DESC
                LIMIT {days}
            """
            stock_data = pd.read_sql_query(query, engine)
            stock_data = correct_nan_val(stock_data, days)
        else:
            query = f"""
                SELECT
                    date,
                    open,
                    high,
                    low,
                    close,
                    volume
                FROM transaction
                WHERE 
                    stock_code = '{stock_code}'
                ORDER BY date
            """
            stock_data = pd.read_sql_query(query, engine)
            stock_data = correct_nan_val(stock_data, None)

        # Get idx of close price
        idx_close = FEATURES.index('close')


        # Extract based on time_span
        stock_data = stock_data[FEATURES].to_numpy()
        x = []
        y = []
        for i in range(len(stock_data) - time_span * 2 + 1):
            x.append(stock_data[i : i + time_span,:])
            y.append(stock_data[i + time_span : i + time_span * 2,idx_close])
        x = np.array(x)
        y = np.array(y)

        for n,feat in enumerate(FEATURES):
            # Pre-extraction for standard normalization
            _x_train = x[:int(0.8 *len(x))]
            # Get the first time series + last value of each examples except the first
            flatten_data = np.concatenate((_x_train[0,:,n].flatten(),_x_train[1:,-1,n].flatten()))
            standard_dict[stock_code][feat] = {
                'mean' : np.mean(flatten_data),
                'std' : np.std(flatten_data)
            }
            # Transform x and y
            # Ignore it for now
            # x[:,:,n] = (x[:,:,n] - standard_dict[stock_code][feat]['mean']) / standard_dict[stock_code][feat]['std']
            # if feat == 'close':
            #     y = (y - standard_dict[stock_code][feat]['mean']) / standard_dict[stock_code][feat]['std']

        # Split data into train, val, and test
        x_train, x_test, y_train, y_test = x[:int(0.8 * len(x))], x[int(0.8 * len(x)):], y[:int(0.8 * len(y))], y[int(0.8 * len(y)):]
        x_train, x_val, y_train, y_val = x_train[:int(0.8 * len(x_train))], x_train[int(0.8 * len(x_train)):], y_train[:int(0.8 * len(y_train))], y_train[int(0.8 * len(y_train)):]

        
        data_dict[stock_code] = {
            'x_train' : x_train,
            'y_train' : y_train,
            'x_val' : x_val,
            'y_val' : y_val,
            'x_test' : x_test,
            'y_test' : y_test
        }
    return data_dict, standard_dict

In [14]:
def le_model(input_shape, time_span = 60):
    inputs = Input(shape = input_shape)

    lstm_freeze = LSTM(8,return_sequences = True, activation = 'relu', kernel_regularizer=regularizers.l2(0.2))(inputs)
    cnn_freeze = Conv1D(filters=8, kernel_size=2, strides=1, padding='same',  activation = 'relu', kernel_regularizer=regularizers.l2(0.2))(lstm_freeze)

    lstm_query = LSTM(8,return_sequences = True)(cnn_freeze)
    lstm_val = LSTM(8,return_sequences = True)(lstm_query)
    attention_lstm = Attention()([lstm_val, lstm_query])
    drop_1 = Dropout(0.1)(attention_lstm)

    cnn_query = Conv1D(filters=8, kernel_size=3, strides=3, padding='same')(drop_1)
    cnn_val = Conv1D(filters=8, kernel_size=3, strides=3, padding='same')(cnn_query)
    attention_cnn = Attention()([cnn_val, cnn_query])
    drop_2 = Dropout(0.1)(attention_cnn)

    lstm_last = LSTM(time_span,return_sequences = False, activation = None, kernel_regularizer=regularizers.l2(0.1))(drop_2)
    outputs = Dense(time_span)(lstm_last)

    model = Model(inputs = inputs, outputs = outputs)

    return model, lstm_freeze, cnn_freeze

In [15]:
# Collect data for transfer learning
base_stocks_flatten = np.unique(np.concatenate([v for _,v in base_stocks.items()]))
transfer_dict,_ = collect_data(base_stocks_flatten, days, time_span)
# Transfer learning using train and validation data
x_train = np.vstack([transfer_dict[stock_code]['x_train'] for stock_code in STOCK_CODES])
y_train = np.vstack([transfer_dict[stock_code]['y_train'] for stock_code in STOCK_CODES])
x_val = np.vstack([transfer_dict[stock_code]['x_val'] for stock_code in STOCK_CODES])
y_val = np.vstack([transfer_dict[stock_code]['y_val'] for stock_code in STOCK_CODES])
x_test = np.vstack([transfer_dict[stock_code]['x_test'] for stock_code in STOCK_CODES])
y_test = np.vstack([transfer_dict[stock_code]['y_test'] for stock_code in STOCK_CODES])

In [10]:
# Create pre training model
pre_trained_model,_,_ = le_model(x_train.shape[1:], time_span)
pre_trained_model.compile(optimizer=Adam(learning_rate=1e-3), loss='mean_squared_error', metrics = ['mae'])
pre_trained_model.summary()

Model: "model"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 60, 5)]      0                                            
__________________________________________________________________________________________________
lstm (LSTM)                     (None, 60, 8)        448         input_1[0][0]                    
__________________________________________________________________________________________________
conv1d (Conv1D)                 (None, 60, 8)        136         lstm[0][0]                       
__________________________________________________________________________________________________
lstm_1 (LSTM)                   (None, 60, 8)        544         conv1d[0][0]                     
______________________________________________________________________________________________

In [11]:
# Pre training the model
early_stopping = EarlyStopping(monitor='val_loss', patience = 10, restore_best_weights=True)

pre_trained_model.fit(
    x_train, 
    y_train, 
    epochs = 1000,
    verbose = True,
    validation_data=(x_val, y_val), 
    callbacks=[early_stopping]
)

y_pred = pre_trained_model.predict(x_test)
rmse = np.sqrt(np.mean(((y_pred - y_test) ** 2)))
print(f"Testing MSE (Normalized): {rmse}")

Epoch 1/1000
10/10 [==============================] - 21s 871ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 2/1000
10/10 [==============================] - 1s 91ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 3/1000
10/10 [==============================] - 1s 82ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 4/1000
10/10 [==============================] - 1s 86ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 5/1000
10/10 [==============================] - 1s 89ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 6/1000
10/10 [==============================] - 1s 94ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 7/1000
10/10 [==============================] - 1s 111ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 8/1000
10/10 [==============================] - 1s 88ms/step - loss: nan - mae: nan - val_loss: nan - val_mae: nan
Epoch 9/1000
10/10 [=========

In [83]:
data_dict, standard_dict = collect_data(STOCK_CODES, days, time_span)
models = {}
for stock_code in STOCK_CODES:
    x_train = data_dict[stock_code]['x_train']
    y_train = data_dict[stock_code]['y_train']
    x_val = data_dict[stock_code]['x_val']
    y_val = data_dict[stock_code]['y_val']
    x_test = data_dict[stock_code]['x_test']
    y_test = data_dict[stock_code]['y_test']

    # Create pre training model
    model,lstm_freeze,cnn_freeze = le_model(x_train.shape[1:], time_span)
    model.compile(optimizer=Adam(learning_rate=1e-3), loss='mean_squared_error')

    # Copy weights from pretrained model
    model.set_weights(pre_trained_model.get_weights())

    # Freeze the weight
    lstm_freeze.trainable = False
    cnn_freeze.trainable = False

    # Pre training the model
    early_stopping = EarlyStopping(monitor='val_loss', patience = 10, restore_best_weights=True)

    model.fit(
        x_train, 
        y_train, 
        epochs = 20,
        verbose = False,
        validation_data=(x_val, y_val), 
        callbacks=[early_stopping]
    )

    y_pred = model.predict(x_test)
    rmse = np.sqrt(np.mean(((y_pred - y_test) ** 2)))
    print(f"{stock_code} - Testing MSE (Normalized): {rmse}")
    models[stock_code] = model

PPT - Testing MSE (Normalized): 5.44387533585461
TPG - Testing MSE (Normalized): 1.2903218163926258


In [84]:
# Create date array
complete_date_df = pd.DataFrame({'date' : pd.date_range(start = 365 * 10, end = pd.to_datetime(datetime.now().date()), freq = 'D')})
complete_date_df = complete_date_df[~complete_date_df['date'].dt.weekday.isin([5,6])]
complete_date = complete_date_df['date'].dt.strftime('%Y-%m-%d')
y_test = data_dict[list(data_dict.keys())[0]]['y_test']
price_length = np.sum(y_test.shape) - 1
complete_date = complete_date.iloc[-price_length:].reset_index(drop = True)
non_weekends_date = []
last_day = pd.to_datetime(complete_date.iloc[-1]).date()
while len(non_weekends_date) <  time_span:
    # Add one day to the current date
    last_day += timedelta(days=1)
    
    # Check if the day is a weekend (Saturday or Sunday)
    if last_day.day < 5:  # Monday to Friday
        non_weekends_date.append(last_day)
complete_date = complete_date.append(pd.Series(non_weekends_date)).reset_index(drop = True)

In [85]:
# Visualize the current prediction and new price in 60 days
for stock_code in STOCK_CODES:
    y_pred = model.predict(x_test)
    # Flip the y
    y_pred_flip = np.flip(y_pred,axis = 1)
    
    pred_norm_price = []
    # Starting from axis 1
    for i in range(y_pred_flip.shape[1] - 1, -1,-1):
        pred_norm_price.append(y_pred_flip.diagonal(i).mean())
    # Starting axis 0
    for i in range(-1, -y_pred_flip.shape[0],-1):
        pred_norm_price.append(y_pred_flip.diagonal(i).mean())
    pred_norm_price = np.array(pred_norm_price)
    # denorm it
    # pred_norm_price = pred_norm_price * standard_dict[stock_code]['close']['std'] + standard_dict[stock_code]['close']['mean']
    # Flatten y_test
    flatten_y_test = np.concatenate([y_test[0,:-1],y_test[:,-1]])
    # Ignore it for now
    # flatten_y_test = flatten_y_test * standard_dict[stock_code]['close']['std'] + standard_dict[stock_code]['close']['mean']

    

    fig = go.Figure(data=[
        go.Scatter(x = complete_date, y = pred_norm_price, name = 'Predict'),
        go.Scatter(x = complete_date, y = flatten_y_test, name = 'True Price'),
    ])
    fig.update_layout(title = stock_code)
    fig.show()